# 1 — Loading data

**Theme:** getting a file off disk and understanding what you get back.

This notebook covers loading a single file, a whole folder, and letting the
package identify the instrument for you — then what a loaded object actually
contains. Plotting is deliberately left out; see
[5 — Plotting](05-plotting.ipynb).

In [ ]:
import aerosoltools as at

## Loading a single file

Every instrument has a loader named `load_<instrument>_file`. They all take a
path and return an object of the appropriate class.

In [ ]:
elpi = at.load_elpi_file("../../tests/data/Sample_ELPI.txt")
elpi

A warning is shown because the particle density recorded in this file is not
1.0 g/cm³. The value is read and stored, but you are told about it because it
affects any mass-based quantity — see
[6 — Dtypes, density and corrections](06-dtypes-density-corrections.ipynb).

## Letting the package identify the instrument

If you do not know (or do not want to hard-code) which instrument produced a
file, `load_file` inspects the contents and dispatches to the right loader.

In [ ]:
data = at.load_file("../../tests/data/Sample_OPS.csv")
type(data).__name__

`detect_instrument` performs the same identification without loading, which is
useful when triaging a folder of mixed exports.

In [ ]:
for name in ["Sample_OPS.csv", "Sample_SMPS.txt", "Sample_DustTrak.csv",
             "Sample_Partector.txt", "Sample_ACSM.csv"]:
    print(f"{name:25s} -> {at.detect_instrument('../../tests/data/' + name)}")

Identification works from the file *contents* first and falls back to the
filename, so renaming a file does not break it. The instruments it knows about
are listed in `INSTRUMENT_LOADERS`, which maps each name to its loader.

In [ ]:
list(at.INSTRUMENT_LOADERS)

## Loading a whole folder

`load_data_from_folder` applies one loader to every matching file in a folder
and concatenates the results. Use `search_word` to restrict which files are
picked up.

In [ ]:
ops = at.load_data_from_folder(
    "../../tests/data/OPS_data",
    at.load_ops_file,
    search_word="OPS",
)

The progress bar and the summary table tell you which files were loaded, which
were skipped, and why. Files are sorted by time, so they do not need to be
passed in order. Only files from the same instrument are combined — the serial
number in the metadata is checked.

To join separate *runs* of one instrument, or to stitch two instruments
covering different size ranges, see
[7 — Combining datasets](07-combining-datasets.ipynb).

## What a loaded object contains

The measurements live in `.data`, indexed by time.

In [ ]:
elpi.data.head()

Anything the loader found that is not a measurement — flows, temperatures,
status flags — goes to `.extra_data`, keeping `.data` clean.

In [ ]:
list(elpi.extra_data.columns)

`.metadata` holds everything the loader learned about the instrument.

In [ ]:
elpi.metadata

The most-used entries also have their own properties.

In [ ]:
print("instrument   :", elpi.instrument)
print("serial number:", elpi.serial_number)
print("unit         :", elpi.unit)
print("dtype        :", elpi.dtype)
print("measurement  :", elpi.measurement)

`unit` and `dtype` describe the primary measurement. Instruments that record
several different quantities report them per column instead, which is what
`column_units` gives you.

In [ ]:
dust = at.load_dusttrak_file("../../tests/data/Sample_DustTrak.csv")
dust.column_units

## Originals and copies

Most operations modify an object in place. `.original_data` always keeps the
data as it was loaded, so you can see what changed.

In [ ]:
before = len(elpi.data)
elpi.timecrop(start="2023-09-07 09:07:00", end="2023-09-07 09:09:00")

print(f"data          : {before} -> {len(elpi.data)} rows")
print(f"original_data : {len(elpi.original_data)} rows (unchanged)")

When you want to branch an analysis without disturbing the original object,
`copy_self` returns an independent deep copy.

In [ ]:
branch = elpi.copy_self()
branch.timecrop(start="2023-09-07 09:07:30", end="2023-09-07 09:08:00")

print(f"branch : {len(branch.data)} rows")
print(f"elpi   : {len(elpi.data)} rows (untouched)")

---

**Next:** [2 — Time adjustments](02-time-adjustments.ipynb) puts two
instruments onto a common time base.